In [0]:
DESCRIBE samples.nyctaxi.trips;

In [0]:
-- Ejercicio 0.1: CTE básica - Estadísticas por zona
WITH estadisticas_por_zona (
    SELECT 
        pickup_zip,
        COUNT(*) AS total_viajes,
        ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
        ROUND(AVG(trip_distance), 2) AS distancia_promedio
    FROM samples.nyctaxi.trips
    WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
    GROUP BY pickup_zip
    HAVING total_viajes > 100
)

SELECT 
  pickup_zip,
  total_viajes,
  tarifa_promedio,
  distancia_promedio
FROM estadisticas_por_zona
ORDER BY total_viajes DESC
LIMIT 20;


In [0]:
-- Ejercicio 0.2: CTE múltiple - Análisis comparativo
WITH promedio_por_hora AS (
    SELECT 
    HOUR(tpep_pickup_datetime) AS hora_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_hora
    FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  GROUP BY HOUR(tpep_pickup_datetime)
),
promedio_general AS(
  SELECT 
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_total
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
)
SELECT 
p.hora_dia,
p.cantidad_viajes,
p.tarifa_promedio_hora,
pg.tarifa_promedio_total,
ROUND(p.tarifa_promedio_hora - pg.tarifa_promedio_total, 2) AS diferencia,
ROUND((p.tarifa_promedio_hora - pg.tarifa_promedio_total) * 100.0 / pg.tarifa_promedio_total, 2) AS porcentaje_diferencia
FROM promedio_por_hora p
CROSS JOIN promedio_general pg
ORDER BY p.hora_dia ASC;

In [0]:
-- Ejercicio 0.3: CTE para limpieza de datos

WITH viajes_validos AS (
    SELECT 
    trip_distance, 
    fare_amount
    FROM samples.nyctaxi.trips 
    WHERE fare_amount > 0 
    AND trip_distance > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
),

estadisticas AS (
    SELECT 
        COUNT(*) AS total_viajes_validos,
        ROUND(MIN(trip_distance), 2) AS minimo,
        ROUND(MAX(trip_distance), 2) AS maximo,
        ROUND(AVG(trip_distance), 2) AS promedio,
        ROUND(PERCENTILE(trip_distance, 0.5), 2) AS distancia_mediana
    FROM viajes_validos
)

SELECT * FROM estadisticas;




In [0]:
-- Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking
SELECT 
ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS viajes_mas_caros,
tpep_pickup_datetime AS fecha,
ROUND(trip_distance, 2) AS distancia,
ROUND(fare_amount, 2) AS tarifa
FROM samples.nyctaxi.trips 
WHERE fare_amount > 0
ORDER BY fare_amount DESC
LIMIT 20; 
